In [ ]:
import json
import os
from pathlib import Path

import requests

In [ ]:
search = search = {
    "query": {
        "function_score": {
            "query": {
                "bool": {
                    "filter": [
                        {"term": {"is_deleted": 0}},
                        {"term": {"is_working": 1}},
                        {"term": {"location_country_iso2": "DE"}},
                        {
                            "multi_match": {
                                "query": "Berlin",
                                "fields": ["location_city", "location_full"],
                            }
                        },
                        {
                            "term": {
                                "active_experience_management_level.exact": "Specialist"
                            }
                        },
                    ],
                    "must": [
                        # Active Title / Department
                        {
                            "bool": {
                                "should": [
                                    {
                                        "match": {
                                            "active_experience_title": {
                                                "query": 'Consultant "Software Engineer" "Product Manager" "Project Coordinator" "Project Manager" "Chief of Staff"',
                                                "operator": "or",
                                            }
                                        }
                                    },
                                    {
                                        "match": {
                                            "active_experience_department": {
                                                "query": "Operations General Management Project Management Engineering Technical Product Consulting",
                                                "operator": "or",
                                            }
                                        }
                                    },
                                ],
                                "minimum_should_match": 1,
                            }
                        },
                        # Education (Degree + Relevant Field)
                        {
                            "nested": {
                                "path": "education",
                                "query": {
                                    "bool": {
                                        "must": [
                                            {
                                                "match": {
                                                    "education.degree": "Bachelor Master BSc MSc"
                                                }
                                            },
                                            {
                                                "multi_match": {
                                                    "query": "software engineering artificial intelligence computer science mathematics data science information systems machine learning",
                                                    "fields": [
                                                        "education.degree^3",
                                                        "education.description^2",
                                                        "education.activities_and_societies",
                                                    ],
                                                    "operator": "or",
                                                }
                                            },
                                        ],
                                        "filter": [
                                            {
                                                "range": {
                                                    "education.date_to_year": {
                                                        "lte": 2026
                                                    }
                                                }
                                            }
                                        ],
                                    }
                                },
                            }
                        },
                        # Languages (English & German)
                        {
                            "nested": {
                                "path": "languages",
                                "query": {
                                    "bool": {
                                        "must": [
                                            {
                                                "match": {
                                                    "languages.language": "English Englisch"
                                                }
                                            },
                                            {
                                                "match": {
                                                    "languages.proficiency": {
                                                        "query": "native bilingual fluent full professional working",
                                                        "operator": "or",
                                                    }
                                                }
                                            },
                                        ]
                                    }
                                },
                            }
                        },
                        {
                            "nested": {
                                "path": "languages",
                                "query": {
                                    "bool": {
                                        "must": [
                                            {
                                                "match": {
                                                    "languages.language": "German Deutsch"
                                                }
                                            },
                                            {
                                                "match": {
                                                    "languages.proficiency": {
                                                        "query": "native bilingual fluent full professional working",
                                                        "operator": "or",
                                                    }
                                                }
                                            },
                                        ]
                                    }
                                },
                            }
                        },
                        # Technical / Core domain match
                        {
                            "bool": {
                                "should": [
                                    {
                                        "multi_match": {
                                            "query": "web application development deployment",
                                            "fields": [
                                                "headline^2",
                                                "summary^3",
                                                "inferred_skills^4",
                                                "historical_skills^2",
                                                "active_experience_description^3",
                                            ],
                                            "minimum_should_match": "50%",
                                        }
                                    },
                                    {
                                        "nested": {
                                            "path": "experience",
                                            "query": {
                                                "multi_match": {
                                                    "query": "web application development deployment",
                                                    "fields": [
                                                        "experience.position_title^2",
                                                        "experience.description^3",
                                                    ],
                                                    "minimum_should_match": "50%",
                                                }
                                            },
                                        }
                                    },
                                    {
                                        "nested": {
                                            "path": "projects",
                                            "query": {
                                                "multi_match": {
                                                    "query": "web application development deployment",
                                                    "fields": [
                                                        "projects.name^2",
                                                        "projects.description^3",
                                                    ],
                                                    "minimum_should_match": "50%",
                                                }
                                            },
                                        }
                                    },
                                ],
                                "minimum_should_match": 1,
                            }
                        },
                    ],
                    "should": [
                        {
                            "multi_match": {
                                "query": "TypeScript",
                                "fields": [
                                    "inferred_skills^6",
                                    "historical_skills^4",
                                    "summary^3",
                                    "active_experience_description^4",
                                ],
                            }
                        },
                        {
                            "multi_match": {
                                "query": "process design automation operational excellence operations workflow",
                                "fields": [
                                    "headline",
                                    "summary^3",
                                    "inferred_skills^3",
                                    "active_experience_description^4",
                                ],
                            }
                        },
                        {
                            "multi_match": {
                                "query": "project management strategic initiatives product launch market expansion stakeholder",
                                "fields": [
                                    "headline^2",
                                    "summary^3",
                                    "inferred_skills^3",
                                    "active_experience_description^4",
                                ],
                            }
                        },
                        {
                            "multi_match": {
                                "query": "financial statements analysis balance sheet income statement cash flow",
                                "fields": [
                                    "summary^3",
                                    "inferred_skills^4",
                                    "active_experience_description^4",
                                ],
                            }
                        },
                        {
                            "nested": {
                                "path": "experience",
                                "query": {
                                    "multi_match": {
                                        "query": "TypeScript process automation operational excellence project management financial statements strategic initiatives product launch market expansion",
                                        "fields": [
                                            "experience.position_title^2",
                                            "experience.department^2",
                                            "experience.description^4",
                                        ],
                                    }
                                },
                            }
                        },
                        {
                            "nested": {
                                "path": "projects",
                                "query": {
                                    "multi_match": {
                                        "query": "TypeScript web application deployment process automation product launch",
                                        "fields": [
                                            "projects.name^2",
                                            "projects.description^4",
                                        ],
                                    }
                                },
                            }
                        },
                        {"exists": {"field": "institution_ranking_score"}},
                    ],
                    "minimum_should_match": 2,
                }
            },
            "functions": [
                {
                    "field_value_factor": {
                        "field": "institution_ranking_score",
                        "factor": 0.15,
                        "modifier": "log1p",
                        "missing": 0,
                    }
                },
                {
                    "field_value_factor": {
                        "field": "profile_score",
                        "factor": 0.1,
                        "modifier": "log1p",
                        "missing": 0,
                    }
                },
            ],
            "score_mode": "sum",
            "boost_mode": "sum",
        }
    }
}

In [ ]:
url = "https://api.coresignal.com/cdapi/v2/employee_multi_source/search/es_dsl"
payload = json.dumps(search)

headers = {
    "Content-Type": "application/json",
    "apikey": os.environ["CORESIGNAL_API_KEY"],
}

response = requests.request("POST", url, headers=headers, data=payload)
person_ids: list[int] = json.loads(response.text)

print(person_ids)

In [ ]:
if 485761352 in person_ids:
    print("Found 485761352")

In [ ]:
with open("../spi/coresiganl_ids.json", "r") as f:
    cached_persons = json.load(f)

In [ ]:
person = person_ids[8]
person

In [ ]:
if person in cached_persons:
    print(f"Person {person} already cached, skipping API call.")
else:
    url = f"https://api.coresignal.com/cdapi/v2/employee_multi_source/collect/{person}"

    headers = {
        "Content-Type": "application/json",
        "apikey": os.environ["CORESIGNAL_API_KEY"],
    }

    response = requests.request("GET", url, headers=headers).json()

    # Write the response to a JSON file
    clean_name = (
        response.get("full_name", "")
        .replace("/", "_")
        .replace("\\", "_")
        .strip()
        .replace(".", "")
        .replace(" ", "_")
        .strip("_")
    ) or f"person_{person}"

    output_file = (
        Path("..")
        / "spi"
        / "coresignal"
        / "employee_multi_source"
        / f"{clean_name}.json"
    )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(response, f, indent=4)

    print(response.get("full_name"))

In [ ]:
cached_persons.append(person)
with open("../spi/coresiganl_ids.json", "w") as f:
    json.dump(sorted(set(cached_persons)), f, indent=4)